# SitePulse — Synthetic Data Simulation Pipeline
### Phase 3: Physics-Based Multi-Trade Construction Project Trajectory Generator

> **Data Integrity Notice:** All datasets generated by this notebook are **synthetic simulations** designed for offline algorithm benchmarking and feature engineering. They do not represent real-world customer deployments or verified industry ground truth.

## 1. Domain Simulation Assumptions & Scenario Topology

Commercial and retail fitouts (30–60 days) follow strict trade precedence:
1. **Civil:** Demolition, screed, masonry (prerequisite for electrical rough-in).
2. **Electrical:** Floor conduits and wall chasing (prerequisite for ceiling carpentry).
3. **Lighting:** Cabling and driver mounting (requires electrical circuit completion).
4. **Carpentry:** Cash counter, display fixtures, partitions (requires civil rough-in).
5. **Painting:** Surface prep, primer, and top coats (dust-sensitive final package).

We model **8 operational scenarios** across 260 distinct projects:
- `on_schedule`: Nominal execution within planned timeline.
- `manpower_delay`: Chronic workforce shortage.
- `material_delay`: Supply disruption holding trade progression.
- `approval_bottleneck`: Client design changes or mall management freezes.
- `concurrent_delays`: Multi-factor compounding delays.
- `recovery_intervention`: Delayed project recovers via contractor workforce surge.
- `misleading_reporting`: Optimism bias & phantom headcounts followed by audit plateau.
- `workforce_fluctuations`: Seasonal / festival absenteeism.

In [ ]:
import os
import pandas as pd
from simulation_engine import run_simulation, FITOUT_TRADES, SCENARIO_TYPES

print(f"Available Trades: {[t['name'] for t in FITOUT_TRADES]}")
print(f"Simulated Scenarios ({len(SCENARIO_TYPES)}): {SCENARIO_TYPES}")

## 2. Execute Simulation Engine (260 Projects, Seed = 42)
Generates relational tables: `projects`, `stores`, `trades`, `daily_updates`, `manpower_log`, `contractor_assignments`, and `delay_events`.

In [ ]:
dfs = run_simulation(n_projects=260, seed=42, output_dir='../data/synthetic')
for name, df in dfs.items():
    print(f"{name}: {len(df):,} records, {len(df.columns)} columns")

## 3. Verify Referential Integrity & Basic Constraints

In [ ]:
projects_df = dfs['projects']
trades_df = dfs['trades']
updates_df = dfs['daily_updates']

# 1. Foreign Key: Every trade belongs to a valid project
orphan_trades = ~trades_df['project_id'].isin(projects_df['id'])
assert orphan_trades.sum() == 0, 'Found orphan trades!'

# 2. Foreign Key: Every daily update belongs to a valid trade
orphan_updates = ~updates_df['trade_id'].isin(trades_df['id'])
assert orphan_updates.sum() == 0, 'Found orphan daily updates!'

# 3. Progress bounds [0, 100]
assert (updates_df['progress_pct'] < 0).sum() == 0, 'Negative progress found!'
assert (updates_df['progress_pct'] > 100).sum() == 0, 'Progress > 100% found!'

# 4. Manpower non-negative
assert (updates_df['workers_present'] < 0).sum() == 0, 'Negative workers present found!'
assert (updates_df['workers_required'] < 0).sum() == 0, 'Negative workers required found!'

# 5. Unique (trade_id, date) constraint
duplicates = updates_df.duplicated(subset=['trade_id', 'date']).sum()
assert duplicates == 0, f'Found {duplicates} duplicate updates!'

print("All referential integrity checks and constraint validations passed cleanly!")

## 4. Scenario Outcome Distribution
Examines how planned deadlines vs actual completions vary by scenario.

In [ ]:
summary = projects_df.groupby('scenario').agg(
    count=('id', 'count'),
    avg_planned_days=('planned_duration_days', 'mean'),
    avg_actual_days=('actual_duration_days', 'mean'),
    avg_delay_days=('delay_days', 'mean'),
    delay_rate_over_3d=('is_delayed_over_3d', 'mean')
).reset_index()

summary['delay_rate_over_3d'] = (summary['delay_rate_over_3d'] * 100).round(1).astype(str) + '%'
summary.round(1)